# 20 — Recorte y alineamiento → Drive

Recorta con `yasma trim` y alinea con `yasma align` (bowtie1 `-v 1 -m 50`), de a
un **proyecto** — organismo + rol. El BAM va a `Drive/10_bam/<org>/<rol>.bam`.

**La unidad es el proyecto entero y no se puede partir.** `yasma align` acumula
la cobertura única de *todas* las librerías del proyecto antes de pesar los
reads multimapeados, así que partirlo en varias llamadas cambia a qué locus va
cada uno. Si la sesión se muere a la mitad de un proyecto, ese proyecto se
rehace entero — pero los que ya terminaron están en Drive y no se tocan.

Por eso §1 mide **antes** de empezar: hay proyectos que no entran en el disco de
una VM de Colab, y eso se sabe en un segundo o a las seis horas.


## Preámbulo: montar Drive y clonar el repo

El repo es público, así que el clon no necesita credenciales. **Los notebooks
llaman a los scripts del repo en vez de reimplementarlos**: el recorte y el
alineamiento tienen que ser los mismos que corren en la máquina local, o dejan
de ser reproducibles.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/tesis')
CLON  = pathlib.Path('/content/tesis')
assert DRIVE.exists(), f'no veo {DRIVE} — ¿montaste la cuenta correcta?'
print('Drive OK:', DRIVE)

In [ ]:
import shutil, subprocess

REPO = 'youkonskernel-afk/tesis'
URL_ANON = 'https://github.com/' + REPO + '.git'

_AYUDA = (
    "No pude clonar de forma anonima y no hay GITHUB_TOKEN en los Secrets.",
    "Dos salidas, cualquiera sirve:",
    "  a) hacer el repo publico: Settings -> General -> Change visibility",
    "  b) crear un PAT de solo lectura y guardarlo como GITHUB_TOKEN en el",
    "     panel de Secrets de Colab (la llave a la izquierda), habilitando",
    "     el acceso para este notebook.",
)


def _sin_token(txt, secreto):
    # git incluye la URL en sus mensajes de error, y esa URL lleva el token.
    return txt.replace(secreto, '***') if secreto else txt


def _actualizar():
    # El clon es un CACHE del repo, no un espacio de trabajo: nada de lo que se
    # escribe durante una corrida vive adentro (el ledger va a Drive). Por eso
    # reset --hard y no pull --ff-only: el pull falla apenas un archivo
    # versionado quede modificado, y fallaba sin hacer ruido, asi que la celda
    # seguia corriendo con el codigo viejo.
    for args in (['fetch', '--depth', '1', 'origin', 'HEAD'],
                 ['reset', '--hard', 'FETCH_HEAD']):
        r = subprocess.run(['git', '-C', str(CLON)] + args,
                           capture_output=True, text=True)
        if r.returncode != 0:
            return False
    return True


def _clonar_de_cero():
    # 1. Anonimo. Alcanza si el repo es publico.
    r = subprocess.run(['git', 'clone', '--depth', '1', URL_ANON, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode == 0:
        return 'clon anonimo (el repo es publico)'

    # 2. Con token de los Secrets de Colab. Para repo privado.
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    if not tok:
        raise RuntimeError(chr(10).join(_AYUDA))

    url = 'https://x-access-token:' + tok + '@github.com/' + REPO + '.git'
    r = subprocess.run(['git', 'clone', '--depth', '1', url, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('el clon con token fallo: ' + _sin_token(r.stderr, tok))

    # Sin esto el token queda escrito en .git/config dentro de la VM.
    subprocess.run(['git', '-C', str(CLON), 'remote', 'set-url', 'origin', URL_ANON],
                   capture_output=True, text=True)
    return 'clon con token (el repo es privado)'


def clonar():
    if CLON.exists():
        if _actualizar():
            return 'clon actualizado'
        # Un clon que no se puede actualizar es peor que no tenerlo: la celda
        # seguiria con codigo viejo sin avisar. Se tira y se clona de nuevo.
        shutil.rmtree(CLON)
    return _clonar_de_cero()


print(clonar())
print(subprocess.run(['git', '-C', str(CLON), 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip())


## Herramientas

Esta etapa sí necesita el entorno completo, a diferencia de los notebooks de
descarga: `bowtie` (que `yasma align` llama directo), `cutadapt` (que `yasma
trim` envuelve), `sra-tools` para volcar los `.sra`, y **ViennaRNA** — sin él
`import RNA` falla en el nivel superior de `hairpin`, que `__init__` importa, y
**no arranca ningún subcomando de YASMA**.

YASMA va pineado por ref de git y no por número: el `pyproject.toml` dentro del
tag `v1.1.1` declara `version = "1.1.0"`, así que la versión instalada no se
puede verificar desde el paquete.


In [ ]:
import glob, os, shutil, subprocess

SRA_VER = '3.1.1'
YASMA_REF = 'git+https://github.com/NateyJay/YASMA@v1.1.1'


def sh(cmd, t=1800):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=t)


def _en_path(ruta):
    if ruta and ruta not in os.environ['PATH']:
        os.environ['PATH'] = ruta + ':' + os.environ['PATH']


# CADA notebook de Colab corre en su propia VM: lo que instalo otro cuaderno no
# existe aca. Esta celda es idempotente; relanzarla no hace nada.
if not (shutil.which('bowtie') and shutil.which('bowtie-build')):
    sh('apt-get -qq update')
    sh('apt-get -qq install -y bowtie')

if not shutil.which('prefetch'):
    c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
    if not c:
        sh(f'wget -q -O /tmp/sra.tar.gz https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/{SRA_VER}/sratoolkit.{SRA_VER}-ubuntu64.tar.gz')
        sh('tar -xzf /tmp/sra.tar.gz -C /opt')
        c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
    _en_path(c[0] if c else '')

if not shutil.which('yasma'):
    sh('pip -q install ViennaRNA cutadapt ' + YASMA_REF)

faltan = [b for b in ('bowtie', 'bowtie-build', 'cutadapt', 'fasterq-dump', 'yasma')
          if not shutil.which(b)]
if faltan:
    raise RuntimeError('no se instalaron: ' + ', '.join(faltan))

# No alcanza con que el binario exista: tiene que ARRANCAR. `import RNA` es el
# fallo clasico y no se ve hasta que se llama a un subcomando.
for cmd, args in [('bowtie', ['--version']), ('cutadapt', ['--version']),
                  ('fasterq-dump', ['--version']), ('yasma', ['--help'])]:
    r = subprocess.run([cmd] + args, capture_output=True, text=True, timeout=120)
    linea = (r.stdout + r.stderr).strip().split('\n')[0]
    print(f"[{'OK ' if r.returncode == 0 else 'MAL'}] {cmd}: {linea[:60]}")


## Configuración

`SRA_DEST` apunta al **mount de Drive**, no a una copia local: los `.sra` no se
bajan a la VM. La regla de `docs/colab.md` es no *escribir* archivos grandes al
FUSE; leerlos está bien, y ahorra ~190 GB de copia.

Lo que sí va al disco de la VM es el recorte y el BAM, que es lo que §1 mide.


In [ ]:
import os, pathlib, shutil, subprocess

PROY_DIR = pathlib.Path('/content/proyectos')     # disco efimero de la VM
GENOMES  = pathlib.Path('/content/genomes')       # idem: el .fna y el indice
BAM_DIR  = pathlib.Path('/content/bams')
for d in (PROY_DIR, GENOMES, BAM_DIR):
    d.mkdir(parents=True, exist_ok=True)

ENV = dict(os.environ)
ENV.update(
    MANIFEST=str(CLON / 'data' / 'srr_manifest.tsv'),
    ADAPTADORES_TSV=str(CLON / 'data' / 'adaptadores.tsv'),
    GENOMAS_LEDGER=str(CLON / 'data' / 'genomas.sha256'),
    SRA_DEST=str(DRIVE / '80_sra'),               # el mount, sin copiar
    GENOMES_DIR=str(GENOMES),
    PROY_DIR=str(PROY_DIR),
    BAM_DIR=str(BAM_DIR),
    CORES=str(os.cpu_count() or 2),
)


def correr(script, *args, env_extra=None, capturar=False):
    """Corre un script del repo. Con capturar=True devuelve (rc, salida) en vez
    de solo rc, para que una celda pueda leer la tabla que el script imprimio
    en vez de pedir que se transcriba a mano."""
    e = dict(ENV, **(env_extra or {}))
    p = subprocess.Popen([str(CLON / 'scripts' / script), *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, env=e, bufsize=1)
    lineas = []
    for ln in p.stdout:
        print(ln, end='')
        if capturar:
            lineas.append(ln)
    rc = p.wait()
    return (rc, ''.join(lineas)) if capturar else rc


libre = shutil.disk_usage('/content').free / 1e9
print(f'disco efímero de la VM: {libre:.0f} GB libres')
print(f'cores: {ENV["CORES"]}')


## 1. Qué entra en esta VM, qué falta, y cuánto tarda

El pico de disco de un proyecto es `recortado + 2 × BAM`: `pysam.sort` escribe
el BAM ordenado **antes** de borrar el sin ordenar, así que los dos conviven.

**Qué falta lo decide Drive, no el disco de la VM.** La VM es efímera: cada
sesión arranca con `/content` vacío, así que `align.sh estado` no puede saber
que un proyecto ya se hizo la semana pasada — diría que falta y lo volvería a
alinear entero. Lo que persiste es `10_bam/<org>/<rol>.bam` en Drive, y ahí
sólo llega lo que pasó `verificar` en §5. Esta celda lo mira y deja
`SIGUIENTE` con el primer proyecto pendiente que entra en el disco; §2 arranca
de ahí en vez de que haya que acordarse de cambiarlo a mano.

Las tres constantes salen de proyectos **medidos**, no de estimaciones:

| constante | medido | acá va | por qué el margen |
| :-- | --: | --: | :-- |
| `B_FQGZ` | 21.8 B/read | 22 | — |
| `B_BAM` | 14.1 B/read | 16 | se midió con **67% de reads sin alinear**, y uno sin alinear ocupa menos que uno colocado |
| `S_POR_M` | 58 s/M read | 60 | genoma de 39 Mb; bowtie sobre 1.1 Gb (`galga`) es más lento por read |

Antes acá decía `B_BAM = 45`, una estimación sin medir: **el pico de BAM
estaba inflado 3×**. Por eso §5 vuelve a imprimir lo medido en cada proyecto —
si alguno se va mucho de estos números, corregilos acá.

`HORAS` no incluye el recorte (§3) ni la subida a Drive (§5).


In [ ]:
import collections, csv, shutil

B_FQGZ   = 22    # bytes por read en .t.fq.gz  (medido 21.8 en sclsc_duplicado)
B_BAM    = 16    # bytes por read en BAM       (medido 14.1, ver tabla arriba)
S_POR_M  = 60    # segundos de reloj por millón de reads (medido 58)
MARGEN   = 5e9   # lo que se deja libre para temporales, indice y el sistema

_ret = {}
with open(CLON / 'data' / 'adaptadores.tsv') as f:
    for r in csv.DictReader((l for l in f if not l.startswith('#')), delimiter='\t'):
        _ret[(r['org'], r['bioproject'])] = float(r['retencion_est'])

proy = collections.defaultdict(lambda: [0, 0.0])
with open(CLON / 'data' / 'srr_manifest.tsv') as f:
    for r in csv.DictReader(f, delimiter='\t'):
        k = (r['org'], r['rol'])
        proy[k][0] += 1
        proy[k][1] += int(r['read_count']) * _ret.get((r['org'], r['bioproject']), 80) / 100

libre = shutil.disk_usage('/content').free
ENTRAN, NO_ENTRAN, HECHOS, SIGUIENTE = [], [], [], None
horas_tot = horas_falta = 0.0
print(f"{'PROYECTO':<18}{'RUNS':>5}{'TRIM_M':>9}{'PICO_GB':>9}{'HORAS':>7}   ESTADO")
for (org, rol), (n, kept) in sorted(proy.items(), key=lambda x: x[1][1]):
    pico = kept * B_FQGZ + 2 * kept * B_BAM
    horas = kept / 1e6 * S_POR_M / 3600
    horas_tot += horas
    entra = pico + MARGEN < libre
    # Lo que persiste entre sesiones es Drive, no /content. Y a 10_bam solo
    # llega lo que paso verificar en §5, asi que un BAM ahi es trabajo hecho.
    hecho = (DRIVE / '10_bam' / org / f'{rol}.bam').exists()
    if hecho:
        HECHOS.append(f'{org}/{rol}')
        estado = 'YA EN DRIVE'
    elif entra:
        ENTRAN.append(f'{org}/{rol}')
        horas_falta += horas
        estado = 'falta, entra'
        if SIGUIENTE is None:
            SIGUIENTE = f'{org}/{rol}'
    else:
        NO_ENTRAN.append(f'{org}/{rol}')
        horas_falta += horas
        estado = 'falta, NO ENTRA en esta VM'
    print(f'{org}_{rol:<10}{n:>5}{kept/1e6:>9.0f}{pico/1e9:>9.1f}{horas:>7.1f}   {estado}')

print(f'\n{len(HECHOS)} ya en Drive, {len(ENTRAN)} faltan y entran, '
      f'{len(NO_ENTRAN)} faltan y no.  Libre: {libre/1e9:.0f} GB')
print(f'Alineamiento de los 18: ~{horas_tot:.0f} h en total, '
      f'~{horas_falta:.0f} h de lo que falta.')
if NO_ENTRAN:
    print('Los que no entran necesitan más disco (Colab Pro) o la máquina local:')
    print('  ' + ' '.join(NO_ENTRAN))
if SIGUIENTE:
    print(f'\nSIGUIENTE = {SIGUIENTE!r}   <- §2 arranca de acá')
else:
    print('\nNo queda ninguno pendiente que entre en esta VM.')


## 2. El genoma

`PROYECTO` sale de `SIGUIENTE`, que §1 calculó mirando qué hay en Drive. No
hace falta acordarse de cambiarlo: si `sclsc/duplicado` ya tiene BAM, §1 pasa
al que sigue. **Para rehacer uno ya hecho, ponelo a mano** — el de Drive se
pisa cuando §5 suba el nuevo.

Verifica el `sha256` del `.gz` contra `data/genomas.sha256` **antes** de
descomprimir, y construye el índice en el disco de la VM.


In [ ]:
PROYECTO = SIGUIENTE       # o '<org>/<rol>' a mano para rehacer uno
assert PROYECTO, 'no hay nada pendiente que entre en esta VM — mirá §1'
print(f'proyecto: {PROYECTO}')

# El .gz se COPIA al disco de la VM en vez de leerse del mount: align.sh
# descomprime al lado del .gz y yasma construye el indice al lado del .fna, y
# eso es escribir varios GB al FUSE de Drive — justo lo que no hay que hacer.
# Son unos cientos de MB por organismo, y se rehace en minutos si la sesion
# muere, asi que no se respalda.
org = PROYECTO.split('/')[0]
shutil.copytree(DRIVE / '70_genomas' / org, GENOMES / org, dirs_exist_ok=True)
correr('align.sh', 'genoma', org)


## 3. Recortar

`PRESUPUESTO_GB` acota cuánto fastq sin recortar hay en disco a la vez; el pico
es ~2× porque la tanda siguiente se vuelca mientras se recorta la actual. Con el
disco de una VM conviene ser conservador.

`verificar` compara la retención medida contra `data/adaptadores.tsv`. Recortar
con la secuencia equivocada **no da error**: deja el `.t.fq.gz` casi vacío, y
por eso esta celda corta ahí en vez de seguir al alineamiento.

**Si corta con `VACIA` o `DESVIADA` en algunas corridas y no en todas**, el
proyecto mezcla kits — pasó en `gadmo/duplicado`, 6 de 12. Seguí con **§3b**,
que mide todas una por una, y **§3c**, que pega las filas y rehace sólo esas.


In [ ]:
PRESUPUESTO_GB = '20'

rc = correr('trim.sh', 'correr', PROYECTO, env_extra={'PRESUPUESTO_GB': PRESUPUESTO_GB})
assert rc == 0, 'el recorte falló'

if correr('trim.sh', 'verificar', PROYECTO) != 0:
    # Cortar esta bien; dejar a la persona adivinando el proximo paso, no.
    print('''
=========================================================================
La retención no da lo que predijo perfil. NO sigas al alineamiento.

Si son ALGUNAS corridas y no todas, el proyecto mezcla kits:
  §3b  mide TODAS las corridas, una por una, y dice cuáles fallaron
  §3c  pega sus filas en data/adaptadores.tsv y las rehace
  §3   otra vez
=========================================================================''')
    raise AssertionError('la retención no da lo que perfil predijo — mirá §3b')


## 3b. Si el recorte no verificó: medir corrida por corrida

`perfil --proyectos` mide **una** corrida de cada proyecto y extiende esa
conclusión a las demás. Cuando un BioProject mezcla kits eso no alcanza:
`gadmo/duplicado` recortó 6 de 12 corridas con la secuencia equivocada y las
otras 6 bien, porque la corrida muestreada era una de las buenas.

Esta celda hace las dos cosas: saca de la tabla de `verificar` **qué** corridas
fallaron —sin transcribirlas a mano— y perfila **todas** las del proyecto, una
por una. Las filas de `PARA data/adaptadores.tsv` vienen con la columna `run`
llena y se pegan tal cual.

Correla sólo si §3 cortó.


In [ ]:
# Las malas salen de la tabla de verificar, no de leerlas y copiarlas: la
# columna CORRIDA es la 3.ª y el veredicto dice VACIA o DESVIADA.
_rc, _sal = correr('trim.sh', 'verificar', PROYECTO, capturar=True)

MALAS = []
for _ln in _sal.splitlines():
    _c = _ln.split()
    if len(_c) > 3 and ('VACIA' in _ln or 'DESVIADA' in _ln):
        MALAS.append(_c[2])

if _rc == 0:
    print('\nverificar pasa: no hay nada que re-medir.')
else:
    print(f'\n{len(MALAS)} corrida(s) fuera de lo esperado: {" ".join(MALAS)}')
    assert MALAS, ('verificar falló pero no pude leer qué corridas — mirá la '
                   'tabla de arriba y poné MALAS = [...] a mano')
    print('\nperfilando TODAS las del proyecto, una por una...\n')
    correr('fetch_runs.sh', 'perfil', '--corridas', PROYECTO, '--tsv')


## 3c. Pegar las filas y rehacer

Del bloque `PARA data/adaptadores.tsv` de arriba, copiá a `FILAS` **sólo** las
corridas cuya medición difiere de la fila del proyecto. La fila `-` del
proyecto sigue valiendo para el resto, así que no hace falta pegar las 12.

`rehacer` es necesario: el recorte es idempotente y las 6 malas ya figuran como
hechas, así que `correr` las saltearía para siempre. Borra su `.t.fq.gz` y las
saca del registro.

Después de esto, volvé a **§3**.


In [ ]:
FILAS = '''
'''.strip()           # pegá acá las filas del bloque de arriba, una por línea

if FILAS:
    _tab = CLON / 'data' / 'adaptadores.tsv'
    _txt = _tab.read_text()
    _nuevas = [l for l in FILAS.splitlines() if l.strip() and l not in _txt]
    if _nuevas:
        _tab.write_text(_txt.rstrip('\n') + '\n' + '\n'.join(_nuevas) + '\n')
    print(f'{len(_nuevas)} fila(s) agregadas a data/adaptadores.tsv')
    print('Acordate de empujarlas con §6: si no, viven sólo en esta VM.')

    assert MALAS, 'no sé qué corridas rehacer — corré §3b primero'
    correr('trim.sh', 'rehacer', PROYECTO, *MALAS)
    print('\nAhora volvé a §3.')
else:
    print('FILAS está vacío: pegá las filas del bloque de §3b.')


## 4. Alinear

Una sola llamada por proyecto, por lo de la cobertura única acumulada. Es la
celda larga: en Colab Free son ~2 vCPU.

Medido en `sclsc_duplicado` (32 M reads, genoma de 39 Mb): **22 min de bowtie
más 4 de `pysam.sort`**, o sea ~49 s por millón de reads. La celda se cronometra
sola y §5 imprime el número de este proyecto, que es con lo que se corrige
`S_POR_M` de §1.


In [ ]:
import time

_t0 = time.monotonic()
rc = correr('align.sh', 'correr', PROYECTO)
T_ALIN = time.monotonic() - _t0
assert rc == 0, 'el alineamiento falló'
print(f'\n§4 tardó {T_ALIN/60:.0f} min de reloj (bowtie + sort)')


## 5. Verificar y subir

**Verificar antes de subir.** Un BAM alineado contra el genoma equivocado se ve
igual que uno bueno: `yasma align` sale con código 0 y 0% alineado. Si
`verificar` falla, el BAM no va a Drive.

Mirá las dos columnas de fracción, que dicen cosas distintas: **ALIN** es lo que
quedó colocado en un locus, **SIN_AL** es lo que no alineó en ninguna parte del
genoma. `SIN_AL` alto no es un genoma repetitivo, son reads que no son de este
genoma — ensamblado equivocado, contaminación, o el huésped si el experimento
es de infección. `verificar` avisa por encima del 50% sin fallar, porque el BAM
está bien escrito: lo que hay que decidir es si esos reads se esperaban.

Al final imprime los bytes/read y los segundos/M read que salieron de verdad,
para ajustar §1.


In [ ]:
rc = correr('align.sh', 'verificar', PROYECTO)
assert rc == 0, 'el BAM no pasó la verificación — no lo subo a Drive'

org, rol = PROYECTO.split('/')
destino = DRIVE / '10_bam' / org
destino.mkdir(parents=True, exist_ok=True)
for suf in ('', '.bai'):
    src = BAM_DIR / org / f'{rol}.bam{suf}'
    if src.exists():
        shutil.copy2(src, destino / src.name)
        print(f'{src.name} -> {destino}  ({src.stat().st_size/1e9:.1f} GB)')

# Lo medido, para corregir las constantes de §1 en vez de arrastrar las de otro
# proyecto. B_BAM llego a estar en 45 sin haberse medido nunca: 3x de mas.
p = PROY_DIR / f'{org}_{rol}'
reads = sum(int(l.split('\t')[4]) for l in
            (p / 'recortadas.tsv').read_text().splitlines()[1:]
            if l.split('\t')[4].isdigit())
fq = sum(f.stat().st_size for f in (p / 'trim').glob('*.t.fq.gz'))
bam = (p / 'align' / 'alignment.bam').stat().st_size
if reads:
    print(f'\nmedido en {org}_{rol}: {reads/1e6:.0f} M reads recortados')
    print(f'  B_FQGZ  {fq/reads:.1f} B/read   (§1 usa {B_FQGZ})')
    print(f'  B_BAM   {bam/reads:.1f} B/read   (§1 usa {B_BAM})')
    try:
        print(f'  S_POR_M {T_ALIN/(reads/1e6):.0f} s/M read  (§1 usa {S_POR_M})')
    except NameError:
        print('  S_POR_M  sin medir: §4 no corrió en esta sesión')
    print('Si alguno difiere mucho, corregilo en §1.')


## 6. Empujar el registro a GitHub

Lo único de esta etapa que va a git: **contra qué ensamblado y con qué
parámetros** se alineó cada proyecto. El BAM va a Drive; esto es texto.

`REVISAR_PRIMERO = True` imprime el diff y no empuja, igual que el dry-run de
`drive_push.sh`. Necesita un `GITHUB_TOKEN` con permiso de escritura en los
Secrets de Colab.


In [ ]:
REVISAR_PRIMERO = True     # False = commitea y empuja

import sys
sys.path.insert(0, str(CLON / 'scripts'))
import colab_git

correr('align.sh', 'ledger', str(CLON / 'data' / 'alineamientos.tsv'))
print()
print(colab_git.empujar(
    CLON, ['data/alineamientos.tsv'],
    f'alineamientos: {PROYECTO} desde Colab',
    revisar=REVISAR_PRIMERO,
    autor=('Colab', 'colab@users.noreply.github.com')))


## 7. El siguiente

Volvé a §2 con otro `PROYECTO` de la lista de §1, de menor a mayor. Cada uno que
termina queda en Drive y no se vuelve a tocar: `align.sh estado` lo dice.

Lo que la VM tiene en `/content` se pierde al cerrar la sesión, y está bien —
los `.t.fq.gz` y el índice se re-generan de los `.sra` y del genoma, que están
en Drive con su checksum.


In [ ]:
correr('align.sh', 'estado')
